### Test cases

- Save test cases

In [1]:
# import json
# import re
# import os

# TESTCASE_FILE = 'message_1.txt'

# with open(TESTCASE_FILE, "r", encoding="utf-8") as f:
#     content = f.read()

# # Regex pattern to match test cases
# pattern = re.compile(r"### Test Case \d+\n\*\*Context\*\*\s*>\s*(.*?)\n\n\*\*Expected Input\*\*\s*>\s*(.*?)\n\n\*\*Reference Summary\*\*\s*>\s*(.*?)\n", re.DOTALL)

# # Extract test cases
# test_cases = []
# matches = pattern.findall(content)
# for match in matches:
#     context, expected_input, reference_summary = match
#     test_cases.append({
#         "id": f"TC{len(test_cases) + 1}",
#         "context": context.strip(),
#         "question": expected_input.strip(),
#         "reference_summary": reference_summary.strip()
#     })

# # Save to JSON file
# output_file = "test_cases.json"
# # # create the file if not found
# # if not os.path.exists(output_file):
# #     open(output_file, 'w').close()

# with open(output_file, "w", encoding="utf-8") as f:
#     json.dump(test_cases, f, indent=4)

# print(f"Test cases extracted and saved to: {output_file}")


### Query LLMs 

- Load test cases

In [2]:
# import json

# test_cases = []

# with open("test_cases.json", "r", encoding="utf-8") as f:
#     test_cases = json.load(f)

# print(test_cases[0]["reference_summary"])

- Run

In [3]:
# from utils import *

# models_list = [
#     'gemma2:2b',
#     'codegemma:2b',
#     'codellama:7b',
#     'codestral',
#     'llama2:13b',
#     'llama3.2:3b',
#     'qwen2.5:0.5b',
#     'qwen2.5:1.5b',
#     'qwen2.5:3b',
#     'qwen2.5:14b',
#     'qwen:1.8b',
#     'qwen:4b',
#     'mistral',
#     'mistrallite',
#     'qwen2.5-coder:1.5b',
#     'deepseek-r1:1.5b',]

# rouge = ROUGE()
# rouge.set_testcases(test_cases)
# for model in models_list:
#     try:
#         rouge.set_model(model)
#         rouge.run_tests()
#     except Exception as e:
#         print(e)

In [4]:
# from langchain_core.output_parsers import StrOutputParser
# from langchain.prompts import PromptTemplate, ChatPromptTemplate
# from langchain_ollama import ChatOllama, OllamaLLM
# from utils import *

# import os

# MODEL_NAME = "gemma2:2b"
# SAVE_FILE_NAME = os.path.join("results", f"{MODEL_NAME.replace(':', '-')}.json")

# model = OllamaLLM(model=MODEL_NAME)

# template = """
# You are a helpful and friendly Next.js assistant. 
# Your responsibility is to answer user queries about Next.js. 
# Answer the question based only and only on the given context below (which got from Next.js documentation). If you can't answer the question, reply "I don't know".

# Context: {context}

# Question: {question}
# """

# prompt = PromptTemplate.from_template(template)

# parser = StrOutputParser()

# chain = prompt | model | parser

# # print(SAVE_FILE_NAME)

In [5]:
# result = []

# for test_case in test_cases:
#     answer = chain.invoke({"context": test_case["context"], "question": test_case["question"]})
#     result.append({
#         "id": test_case["id"],
#         "answer": answer
#     })

# with open(SAVE_FILE_NAME, "w", encoding="utf-8") as f:
#     json.dump(result, f, indent=4)

### Evaluate answers

In [6]:
# import json

# # Load test cases
# test_cases = []
# with open("test_cases.json", "r", encoding="utf-8") as f:
#     test_cases = json.load(f)

# # Load results
# results = []
# with open("results/codegemma-2b.json", "r", encoding="utf-8") as f:
#     results = json.load(f)

# # Create a dictionary from results using 'id' for quick lookup
# results_dict = {item["id"]: item for item in results}

# # Merge the test cases with corresponding results
# merged_data = []
# for test_case in test_cases:
#     case_id = test_case["id"]
#     if case_id in results_dict:
#         merged_data.append({
#             "id": case_id,
#             "answer": results_dict[case_id]["answer"],  # Get the answer from results
#             "reference_summary": test_case["reference_summary"]  # Keep reference summary from test cases
#         })

# # Save the merged data to a new JSON file
# # with open("/mnt/data/merged_test_cases.json", "w", encoding="utf-8") as f:
# #     json.dump(merged_data, f, indent=4, ensure_ascii=False)

# # print("Merged JSON file saved as merged_test_cases.json")

# merged_data


- Load test cases

In [1]:
# import os

# models_list = os.listdir('results')
# models_list

models_list = [
 'codegemma:2b',
 'codellama:7b',
 'codestral:22b',
 'deepseek-r1:1.5b',
 'gemma2:2b',
 'llama2:13b',
 'llama3.2:3b',
 'mistral:7b',
 'mistrallite:7b',
 'qwen:1.8b',
 'qwen:4b',
 'qwen2.5:0.5b',
 'qwen2.5:1.5b',
 'qwen2.5:14b',
 'qwen2.5:3b',
 'qwen2.5-coder:1.5b']


In [2]:
from rouge_utils import ROUGE
import json

rouge = ROUGE()
rouge_results = []
for model in models_list:
    try:
        rouge.set_model(model)
        rouge.merge_results(output_file=f"results/{model.replace(':', '-')}.json", test_cases_file="test_cases.json")
        scores = rouge.score_rouge()
        # print(scores)
        rouge_results.append({
            "model": model,
            "scores": scores
        })
    except Exception as e:
        print(e)
    print(model, 'done')
    

with open("results/summary_123L.json", "w", encoding="utf-8") as f:
    json.dump(rouge_results, f, indent=4)

codegemma:2b done
codellama:7b done
codestral:22b done
deepseek-r1:1.5b done
gemma2:2b done
llama2:13b done
llama3.2:3b done
mistral:7b done
mistrallite:7b done
qwen:1.8b done
qwen:4b done
qwen2.5:0.5b done
qwen2.5:1.5b done
qwen2.5:14b done
qwen2.5:3b done
qwen2.5-coder:1.5b done
